# IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import BallTree

from google.colab import drive
drive.mount('/content/drive', force_remount= True) #Use path /content/drive/MyDrive/Aerosol/files/ to access files

Mounted at /content/drive


# **MERGE MCD12C1 L3 to AERONET**

In [ ]:
aeronet= pd.read_csv('/content/drive/MyDrive/Aerosol/files/aeronet.csv')
aeronet

In [ ]:
mcd = pd.read_csv('/content/drive/MyDrive/Aerosol/files/MCD12C1_L3_cleaned/MCD12C1.A2021001.061.2022217040006.csv')

In [ ]:
mcd

In [ ]:
import pandas as pd
from scipy.spatial import cKDTree

def merge_mcd(df, mcd):
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df['Year'] = df['Datetime'].dt.year

    mcd = mcd.copy()
    mcd['Year'] = mcd['Year'].astype(int)
    df['Year'] = df['Year'].astype(int)

    results = []

    for year in sorted(df['Year'].unique()):
        df_year = df[df['Year'] == year].reset_index(drop=True)
        mcd_year = mcd[mcd['Year'] == year].reset_index(drop=True)

        if mcd_year.empty or df_year.empty:
            continue

        tree = cKDTree(mcd_year[['Latitude', 'Longitude']].values)
        dist, idx = tree.query(df_year[['Latitude(Degrees)', 'Longitude(Degrees)']].values)

        matched_mcd = mcd_year.iloc[idx].reset_index(drop=True)

        merged = pd.concat([df_year.reset_index(drop=True), matched_mcd[['LC_IGBP', 'Pct_Agreement']]], axis=1)
        results.append(merged)

    merged_df = pd.concat(results, ignore_index=True)
    return merged_df

aeronet_mcd= merge_mcd(aeronet, mcd)

aeronet_mcd.to_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd.csv', index=False)
aeronet_mcd

In [ ]:
aeronet_mcd

# **MERGE TROPOMI**

In [ ]:
tropomi = [pd.read_csv('/content/drive/MyDrive/Aerosol/files/Tropomi-2021-01-01-2021-01-15.csv'),
          pd.read_csv('/content/drive/MyDrive/Aerosol/Aaditya/Tropomi-2021-01-16-2021-01-31.csv'),
          pd.read_csv('/content/drive/MyDrive/Aerosol/Siddh/Tropomi-2021-02-01-2021-02-15.csv'),
          pd.read_csv('/content/drive/MyDrive/Aerosol/Sarthi/Tropomi-2021-02-16-2021-02-28.csv')]

In [ ]:
aeronet_mcd.columns

In [ ]:
for i in range(len(tropomi)):
    print(tropomi[i].shape)

In [ ]:
from datetime import timedelta
from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd
from tqdm import tqdm

# Ensure datetime columns are timezone-naive
def merge_tropomi(df, tropomi):
    df["Datetime"] = pd.to_datetime(df["Datetime"]).dt.tz_localize(None)
    tropomi["datetime"] = pd.to_datetime(tropomi["datetime"]).dt.tz_localize(None)

    # Precompute radians for TROPOMI coords
    tropomi_coords_rad = np.radians(tropomi[["latitude", "longitude"]].values)
    tropomi_times = tropomi["datetime"].values

    # Build BallTree for spatial search
    tree = BallTree(tropomi_coords_rad, metric="haversine")

    # Radius in radians
    radius_km = 50
    radius = radius_km / 6371

    merged_rows = []

    aeronet_coords_rad = np.radians(df[["Latitude(Degrees)", "Longitude(Degrees)"]].values)
    aeronet_times = df["Datetime"].values

    # Vectorized loop
    for idx in tqdm(range(len(df)), desc="Merging"):
        aeronet_point = aeronet_coords_rad[idx].reshape(1, -1)
        aeronet_time = aeronet_times[idx]

        # Query spatial neighbors
        indices = tree.query_radius(aeronet_point, r=radius)[0]
        if len(indices) == 0:
            merged_rows.append({**df.iloc[idx].to_dict(), **{col: np.nan for col in tropomi.columns}})
            continue

        # Filter by time difference
        candidate_times = tropomi_times[indices]
        time_deltas = np.abs((candidate_times - aeronet_time).astype('timedelta64[s]')) / np.timedelta64(1, 's')

        within_time = time_deltas <= 5 * 3600
        if not np.any(within_time):
            merged_rows.append({**df.iloc[idx].to_dict(), **{col: np.nan for col in tropomi.columns}})
            continue

        best_idx = indices[within_time][np.argmin(time_deltas[within_time])]
        merged = {**df.iloc[idx].to_dict(), **tropomi.iloc[best_idx].to_dict()}
        merged_rows.append(merged)

    final = pd.DataFrame(merged_rows)
    aeronet_cols = df.columns.tolist()
    tropomi_cols = [col for col in final.columns if col not in aeronet_cols]
    final = final[aeronet_cols + tropomi_cols]
    return final

aeronet_mcd_tropomi= aeronet_mcd
for i in range(len(tropomi)):
    aeronet_mcd_tropomi = merge_tropomi(aeronet_mcd_tropomi, tropomi[i])

aeronet_mcd_tropomi.to_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd_tropomi.csv', index=False)
aeronet_mcd_tropomi

In [ ]:
tropomi_cols = tropomi[0].columns.tolist()
aeronet_mcd_tropomi.dropna(subset=tropomi_cols, how='all', inplace=True)
aeronet_mcd_tropomi.to_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd_tropomi.csv', index=False)
aeronet_mcd_tropomi

# **MERGE MYD04 L2**

In [ ]:
myd = pd.read_csv('/content/drive/MyDrive/Aerosol/files/MYD04.csv')
myd['Datetime'] = pd.to_datetime(
    myd['Date'] + ' ' + myd['Time'],
    format='%Y-%m-%d %H:%M:%S'
)
myd

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from datetime import timedelta

def merge_myd(df, myd):
    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    myd['Datetime'] = pd.to_datetime(myd['Datetime'])

    df['Date'] = df['Datetime'].dt.date
    myd['Date'] = myd['Datetime'].dt.date

    results = []

    common_dates = set(df['Date']).intersection(set(myd['Date']))
    for date in sorted(common_dates):
        df1 = df[df['Date'] == date].copy()
        df2 = myd[myd['Date'] == date].copy()

        if df1.empty or df2.empty:
            continue

        df1 = df1.reset_index(drop=True)
        df2 = df2.reset_index(drop=True)

        nearest_indices = np.abs(
            df1["Datetime"].values[:, None] - df2["Datetime"].values[None, :]
        ).argmin(axis=1)

        df2_nearest_times = df2.iloc[nearest_indices].reset_index(drop=True)

        time_diff = (df1['Datetime'] - df2_nearest_times['Datetime']).abs()
        within_1_hour_mask = time_diff <= timedelta(hours=1)

        df1 = df1[within_1_hour_mask].reset_index(drop=True)
        df2_nearest_times = df2_nearest_times[within_1_hour_mask].reset_index(drop=True)

        if df1.empty:
            continue

        tree = cKDTree(df2_nearest_times[["Latitude", "Longitude"]].values)
        distances, indices = tree.query(df1[["Latitude(Degrees)", "Longitude(Degrees)"]].values)

        df2_matched = df2_nearest_times.iloc[indices].reset_index(drop=True)

        merged = pd.concat([df1, df2_matched.add_suffix('_MYD')], axis=1)
        results.append(merged)

    df_myd = pd.concat(results, ignore_index=True)
    return df_myd

aeronet_mcd_tropomi_myd = merge_myd(aeronet_mcd_tropomi, myd)
aeronet_mcd_tropomi_myd.to_csv('/content/drive/MyDrive/Aerosol/files/aeronet_mcd_tropomi_myd.csv', index=False)
aeronet_mcd_tropomi_myd

# **MERGE TROPOMI & MODIS FOR 1/1/2021**

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

from google.colab import drive
drive.mount('/content/drive', force_remount= True)

In [ ]:
myd= pd.read_csv('/content/drive/MyDrive/Aerosol/files/MYD04.csv')
myd

In [ ]:
myd['Date'] = pd.to_datetime(myd['Date']).dt.date
myd = myd[myd['Date'] == dt.date(2021, 1, 1)]
myd

In [ ]:
tropomi = pd.read_csv('/content/drive/MyDrive/Aerosol/files/Tropomi-2021-01-01-2021-01-15.csv')
tropomi

In [ ]:
tropomi= tropomi[(tropomi['year']==2021) & (tropomi['month']==1) & (tropomi['day']==1)]
tropomi

In [ ]:
from sklearn.neighbors import BallTree

# Read files with .copy() to avoid SettingWithCopyWarning
myd = myd.copy()
tropomi = tropomi.copy()

tropomi_coords_rad = np.radians(tropomi[['latitude', 'longitude']].values)
tree = BallTree(tropomi_coords_rad, leaf_size=15, metric='haversine')

# Step 3: Function to find the closest TROPOMI point for one MODIS point
def find_closest_index(lat, lon):
    point_rad = np.radians([[lat, lon]])
    dist, ind = tree.query(point_rad, k=1)
    return ind[0][0]

# Step 4: Loop through each row of MODIS and find closest TROPOMI point
tropomi_indices = []
for idx, row in myd.iterrows():
    lat = row['Latitude']
    lon = row['Longitude']
    index = find_closest_index(lat, lon)
    tropomi_indices.append(index)

# Step 5: Add index to MODIS DataFrame
myd['tropomi_index'] = tropomi_indices

# Step 6: Create a row index for TROPOMI to merge on
tropomi = tropomi.reset_index(drop=True)
tropomi['tropomi_index'] = tropomi.index

# Step 7: Merge MODIS with corresponding TROPOMI rows
merged_df = pd.merge(myd, tropomi, on='tropomi_index', how='inner')

# Step 8: Done — preview
# merged_df.to_csv('/content/drive/MyDrive/Aerosol/files/myd_tropomi.csv', index=False)
merged_df

In [ ]:
mcd= pd.read_csv('/content/drive/MyDrive/Aerosol/files/MCD12C1_L3_cleaned/MCD12C1.A2021001.061.2022217040006.csv')

In [ ]:
mcd.columns

In [ ]:
merged_df.columns

In [ ]:
from sklearn.neighbors import BallTree

modis_coords = np.radians(merged_df[['Latitude', 'Longitude']].values)
mcd_coords = np.radians(mcd[['Latitude', 'Longitude']].values)

tree = BallTree(mcd_coords, metric='haversine')

mcd_indices = []
for row in merged_df.itertuples(index=False):
    point_rad = np.radians([[row.Latitude, row.Longitude]])
    dist, ind = tree.query(point_rad, k=1)
    mcd_indices.append(ind[0][0])

merged_df['mcd_index'] = mcd_indices

mcd = mcd.reset_index(drop=True)
mcd['mcd_index'] = mcd.index

# Step 7: Merge on the index
final_df = merged_df.merge(mcd, on='mcd_index', how='inner')

# Step 8: Clean up
final_df.drop(columns=['mcd_index'], inplace=True)

# ✅ Done
# final_df.to_csv('/content/drive/MyDrive/Aerosol/files/final_merged_with_mcd.csv', index=False)
final_df.head()

In [ ]:
final_df.columns

In [ ]:
columns_to_drop = ['tropomi_index', 'index', 'latitude', 'longitude', 'year', 'month', 'day', 'time', 'datetime', 'Longitude_y', 'Latitude_y', 'Year']
final_df = final_df.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
final_df = final_df.rename(columns={'Latitude_x': 'Latitude', 'Longitude_x': 'Longitude'})
final_df

In [ ]:
final_df.to_csv('/content/drive/MyDrive/Aerosol/files/myd_tropomi_mcd.csv', index=False)

In [ ]:
tropomi.columns

In [ ]:
from sklearn.neighbors import BallTree

tropomi_coords = np.radians(tropomi[['latitude', 'longitude']].values)
mcd_coords = np.radians(mcd[['Latitude', 'Longitude']].values)

tree = BallTree(mcd_coords, metric='haversine')

mcd_indices = []
for row in tropomi.itertuples(index=False):
    lat, lon = row.latitude, row.longitude
    point_rad = np.radians([[lat, lon]])
    dist, ind = tree.query(point_rad, k=1)
    mcd_indices.append(ind[0][0])

tropomi['mcd_index'] = mcd_indices

mcd = mcd.reset_index(drop=True)
mcd['mcd_index'] = mcd.index

tropomi_mcd_merged = tropomi.merge(mcd, on='mcd_index', how='inner')

tropomi_mcd_merged.drop(columns=['mcd_index'], inplace=True)

# ✅ Result
# tropomi_mcd_merged.to_csv('/content/drive/MyDrive/Aerosol/files/tropomi_with_mcd.csv', index=False)
tropomi_mcd_merged

In [ ]:
columns_to_drop = ['index', 'year', 'month', 'day', 'time', 'tropomi_index', 'Longitude', 'Latitude', 'Year']
tropomi_mcd_merged = tropomi_mcd_merged.drop(columns=columns_to_drop, errors='ignore')
tropomi_mcd_merged= tropomi_mcd_merged.rename(columns={'latitude': 'Latitude', 'longitude': 'Longitude'})
print(tropomi_mcd_merged.columns)
tropomi_mcd_merged['Date'] = pd.to_datetime(tropomi_mcd_merged['datetime']).dt.date
tropomi_mcd_merged['Time'] = pd.to_datetime(tropomi_mcd_merged['datetime']).dt.time
tropomi_mcd_merged.drop(columns=['datetime'], inplace=True)
tropomi_mcd_merged= tropomi_mcd_merged[['Date', 'Time', 'Latitude', 'Longitude', 'CO', 'AER_AI', 'NO2', 'SZA', 'LC_IGBP', 'Pct_Agreement']]
tropomi_mcd_merged.to_csv('/content/drive/MyDrive/Aerosol/files/tropomi_mcd.csv', index=False)
tropomi_mcd_merged